In [0]:
import mlflow
import pandas as pd
from src.config import MODEL_NAME, FEATURE_TABLE, PRED_TABLE

model = mlflow.pyfunc.load_model(
    f"models:/{MODEL_NAME}@prod"
)

df = spark.table(FEATURE_TABLE).toPandas()

X = pd.get_dummies(df.drop("Exited", axis=1))

df["churn_score"] = model.predict(X)

df["risk_segment"] = df["churn_score"].apply(
    lambda x: "HIGH RISK" if x == 1 else "LOW RISK"
)

spark.createDataFrame(df).write.mode("overwrite").saveAsTable(PRED_TABLE)

print("Inference done from the Production Model")